# DENTRAT — Fine-Tune v3 (Roboflow COCO Dataset)

Fine-tune **dental_model_v2.pth** on the Roboflow panoramic X-ray dataset.

**Output:** `dental_model_v3.pth` (auto-downloaded)

## Steps
1. Runtime → **GPU (T4)**
2. Run all cells
3. Paste Roboflow API key when prompted
4. Upload **dental_model_v2.pth** when prompted
5. **Review class mapping** in Section 4 — edit `MANUAL_OVERRIDES` if needed
6. Wait ~1–2 hours → download **dental_model_v3.pth**

Deploy v3 on Railway: upload to `models/dental_model_v3.pth` (backend prefers v3 over v2).

## 1. Install Dependencies

In [ ]:
!pip install -q roboflow albumentations opencv-python-headless scikit-learn matplotlib pandas

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Download Roboflow COCO Dataset

In [ ]:
import os
from getpass import getpass
from roboflow import Roboflow

WORK_DIR = "/content/dentrat_v3_training"
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Paste your Roboflow API key (from Roboflow → Settings → API key)
ROBOFLOW_API_KEY = getpass("Enter Roboflow API key: ")

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("opg-unbmz").project("dental-x-ray-panoramic")
version = project.version(2)
dataset = version.download("coco")

DATASET_DIR = dataset.location if hasattr(dataset, "location") else str(dataset)
print(f"Dataset downloaded to: {DATASET_DIR}")

for split in ["train", "valid", "test"]:
    split_dir = os.path.join(DATASET_DIR, split)
    if os.path.isdir(split_dir):
        n_imgs = len([f for f in os.listdir(split_dir) if f.lower().endswith(('.jpg','.jpeg','.png'))])
        print(f"  {split}: {n_imgs} images")

## 3. Configuration & Target Classes

Your model uses **7 anomaly classes** (IDs 1–7) + background (0) = **8 total** in Faster R-CNN.

In [ ]:
import json
import re
import copy
import time
import glob
import numpy as np
import pandas as pd
import cv2
import albumentations as A
from albumentations.pytorch import ToTensorV2
from collections import defaultdict
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

CLASS_NAMES = {
    1: "Caries",
    2: "Impacted Teeth",
    3: "Broken Down Crown/Root",
    4: "Infection",
    5: "Fractured Teeth",
    6: "Periodontal Bone Loss",
    7: "Other Abnormalities",
}

NUM_CLASSES = 8  # background + 7 classes
IMAGE_SIZE = 416
BATCH_SIZE = 4
NUM_EPOCHS = 25
LR_BACKBONE = 0.00005
LR_HEAD = 0.0005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Keyword-based auto-mapping (case-insensitive)
KEYWORD_MAP = {
    1: ["caries", "cavity", "cavities", "decay", "dental caries"],
    2: ["impacted", "impaction", "impacted tooth", "impacted teeth"],
    3: ["broken crown", "broken down", "crown", "root", "broken-down"],
    4: ["infection", "abscess", "infected"],
    5: ["fractured", "fracture", "crack", "fractured tooth"],
    6: ["periodontal", "bone loss", "bone-loss", "periodontitis"],
    7: ["other", "abnormal", "anomaly", "misc"],
}
SKIP_KEYWORDS = ["healthy", "normal", "no finding", "background", "none"]

# Numeric ID fallback (same as Dataset_Batch2 mapping)
NUMERIC_ID_MAP = {
    0: None, 1: 1, 2: 2, 3: 4, 4: 5, 5: 3, 6: 6, 7: 7, 8: None,
}

print(f"Training on: {DEVICE}")

## 4. Auto-Detect & Map Dataset Classes

**Review the mapping table below.** If any class is wrong or unmapped, edit `MANUAL_OVERRIDES` and re-run this cell.

In [ ]:
def find_coco_json(split_dir):
    """Find COCO annotation JSON in a split folder."""
    patterns = [
        "_annotations.coco.json",
        "instances_default.json",
        "*.coco.json",
        "_annotations.json",
    ]
    for pat in patterns:
        matches = glob.glob(os.path.join(split_dir, pat))
        if matches:
            return matches[0]
    return None


def load_categories(dataset_dir):
    """Load unique categories from train COCO JSON."""
    train_dir = os.path.join(dataset_dir, "train")
    json_path = find_coco_json(train_dir)
    if not json_path:
        raise FileNotFoundError(f"No COCO JSON found in {train_dir}")
    with open(json_path) as f:
        data = json.load(f)
    return {cat["id"]: cat["name"] for cat in data.get("categories", [])}, json_path


def auto_map_category(cat_id, cat_name):
    """Map a Roboflow category to target class 1-7 or None (skip)."""
    name_lower = cat_name.lower().strip()
    for skip in SKIP_KEYWORDS:
        if skip in name_lower:
            return None
    for target_id, keywords in KEYWORD_MAP.items():
        for kw in keywords:
            if kw in name_lower or name_lower in kw:
                return target_id
    # Fallback: try numeric category id mapping
    if cat_id in NUMERIC_ID_MAP:
        return NUMERIC_ID_MAP[cat_id]
    try:
        n = int(re.sub(r"\D", "", name_lower) or -1)
        if n in NUMERIC_ID_MAP:
            return NUMERIC_ID_MAP[n]
    except Exception:
        pass
    return "UNMAPPED"


raw_categories, COCO_JSON_SAMPLE = load_categories(DATASET_DIR)
print(f"Found {len(raw_categories)} categories in dataset:\n")
print(f"{'ID':<6} {'Roboflow Name':<35} {'→ Target':<25}")
print("-" * 70)

# ── EDIT THIS if auto-mapping is wrong ──
# Format: { roboflow_category_id: target_class_id or None to skip }
# Example: { 9: 1, 10: None }
MANUAL_OVERRIDES = {}

CATEGORY_MAP = {}  # roboflow_cat_id → target 1-7 or None
for cat_id, cat_name in sorted(raw_categories.items()):
    if cat_id in MANUAL_OVERRIDES:
        mapped = MANUAL_OVERRIDES[cat_id]
    else:
        mapped = auto_map_category(cat_id, cat_name)
    CATEGORY_MAP[cat_id] = mapped if mapped != "UNMAPPED" else None
    if mapped == "UNMAPPED":
        target_str = "⚠ UNMAPPED — add to MANUAL_OVERRIDES"
    elif mapped is None:
        target_str = "SKIP"
    else:
        target_str = f"Class {mapped}: {CLASS_NAMES.get(mapped, '?')}"
    print(f"{cat_id:<6} {cat_name:<35} {target_str}")

unmapped = [raw_categories[k] for k, v in CATEGORY_MAP.items() if auto_map_category(k, raw_categories[k]) == "UNMAPPED" and k not in MANUAL_OVERRIDES]
if unmapped:
    print(f"\n⚠ Warning: {len(unmapped)} unmapped class(es): {unmapped}")
    print("Edit MANUAL_OVERRIDES above and re-run this cell before training.")
else:
    print("\n✓ All classes mapped successfully.")

## 5. COCO Dataset Loader

In [ ]:
class CocoDentalDataset(Dataset):
    """Load Roboflow COCO exports for Faster R-CNN training."""

    def __init__(self, dataset_dir, split, category_map, transforms=None):
        self.images_dir = os.path.join(dataset_dir, split)
        self.transforms = transforms
        self.category_map = category_map

        json_path = find_coco_json(self.images_dir)
        if not json_path:
            raise FileNotFoundError(f"No COCO JSON in {self.images_dir}")

        with open(json_path) as f:
            coco = json.load(f)

        self.id_to_file = {img["id"]: img["file_name"] for img in coco["images"]}
        self.annotations = defaultdict(list)
        skipped = 0

        for ann in coco["annotations"]:
            src_id = ann["category_id"]
            target = category_map.get(src_id)
            if target is None:
                skipped += 1
                continue
            x, y, w, h = ann["bbox"]
            if w <= 1 or h <= 1:
                continue
            self.annotations[ann["image_id"]].append({
                "bbox": [x, y, x + w, y + h],  # Pascal VOC for albumentations
                "label": target,
            })

        self.image_ids = [iid for iid in self.id_to_file if len(self.annotations.get(iid, [])) > 0]
        print(f"  [{split}] {len(self.image_ids)} images, skipped {skipped} annotations")

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_id = self.image_ids[idx]
        filename = self.id_to_file[img_id]
        path = os.path.join(self.images_dir, filename)
        image = cv2.imread(path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        anns = self.annotations[img_id]
        bboxes = [a["bbox"] for a in anns]
        labels = [a["label"] for a in anns]

        if self.transforms:
            t = self.transforms(image=image, bboxes=bboxes, class_labels=labels)
            image = t["image"]
            bboxes = t["bboxes"]
            labels = t["class_labels"]

        if not isinstance(image, torch.Tensor):
            image = torch.from_numpy(image.transpose(2, 0, 1)).float() / 255.0

        boxes = torch.as_tensor(bboxes, dtype=torch.float32)
        labels_t = torch.as_tensor(labels, dtype=torch.int64)
        return image, {"boxes": boxes, "labels": labels_t, "image_id": torch.tensor([idx])}


def collate_fn(batch):
    return tuple(zip(*batch))


def get_train_transforms():
    return A.Compose([
        A.RandomRotate90(p=0.5),
        A.HorizontalFlip(p=0.5),
        A.Affine(scale=(0.85, 1.15), rotate=(-30, 30), p=0.7),
        A.RandomBrightnessContrast(p=0.5),
        A.GaussNoise(p=0.3),
        A.ElasticTransform(alpha=1, sigma=50, p=0.3),
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["class_labels"], min_visibility=0.3))


def get_val_transforms():
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ], bbox_params=A.BboxParams(format="pascal_voc", label_fields=["class_labels"]))


print("Loading datasets...")
train_dataset = CocoDentalDataset(DATASET_DIR, "train", CATEGORY_MAP, get_train_transforms())
valid_dataset = CocoDentalDataset(DATASET_DIR, "valid", CATEGORY_MAP, get_val_transforms())
test_dataset  = CocoDentalDataset(DATASET_DIR, "test",  CATEGORY_MAP, get_val_transforms())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2, pin_memory=True)

## 6. Upload & Load dental_model_v2.pth

Upload your current best model to fine-tune from.

In [ ]:
from google.colab import files

print("Upload dental_model_v2.pth:")
uploaded = files.upload()
PRETRAINED_PATH = list(uploaded.keys())[0]
print(f"Loaded: {PRETRAINED_PATH} ({os.path.getsize(PRETRAINED_PATH)/1e6:.1f} MB)")


def build_model(num_classes=NUM_CLASSES):
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model


def load_pretrained(model_path):
    model = build_model()
    ckpt = torch.load(model_path, map_location=DEVICE, weights_only=False)
    state = ckpt.get("model_state_dict") or ckpt.get("state_dict") or ckpt
    model_dict = model.state_dict()
    pretrained = {k: v for k, v in state.items() if k in model_dict and v.shape == model_dict[k].shape}
    model_dict.update(pretrained)
    model.load_state_dict(model_dict)
    print(f"Loaded {len(pretrained)}/{len(model_dict)} weight tensors from v2 model")
    return model

model = load_pretrained(PRETRAINED_PATH)
model.to(DEVICE)

## 7. Training (25 epochs, LR 0.0005 head / 0.00005 backbone)

In [ ]:
def get_optimizer(model):
    params_backbone, params_head = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        (params_backbone if "backbone" in name else params_head).append(p)
    return torch.optim.SGD([
        {"params": params_backbone, "lr": LR_BACKBONE},
        {"params": params_head, "lr": LR_HEAD},
    ], momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)


def train_one_epoch(model, loader, optimizer, epoch):
    model.train()
    total_loss, n_batches, skipped = 0.0, 0, 0
    for images, targets in loader:
        valid = [(img, tgt) for img, tgt in zip(images, targets) if len(tgt["boxes"]) > 0]
        if not valid:
            skipped += 1
            continue
        imgs = [i.to(DEVICE) for i, _ in valid]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for _, t in valid]
        loss_dict = model(imgs, tgts)
        loss = sum(loss_dict.values())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        n_batches += 1
    avg = total_loss / max(n_batches, 1)
    print(f"  Epoch {epoch} Train Loss: {avg:.4f} (skipped {skipped} empty batches)")
    return avg


@torch.no_grad()
def eval_loss(model, loader):
    model.train()
    total, n = 0.0, 0
    for images, targets in loader:
        valid = [(img, tgt) for img, tgt in zip(images, targets) if len(tgt["boxes"]) > 0]
        if not valid:
            continue
        imgs = [i.to(DEVICE) for i, _ in valid]
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for _, t in valid]
        total += sum(model(imgs, tgts).values()).item()
        n += 1
    return total / max(n, 1)


optimizer = get_optimizer(model)
train_losses, val_losses = [], []
best_val, best_state = float("inf"), None

print("Starting fine-tuning...")
t0 = time.time()
for epoch in range(1, NUM_EPOCHS + 1):
    tl = train_one_epoch(model, train_loader, optimizer, epoch)
    vl = eval_loss(model, valid_loader)
    train_losses.append(tl)
    val_losses.append(vl)
    print(f"  Epoch {epoch} Val Loss: {vl:.4f}")
    if vl < best_val:
        best_val = vl
        best_state = copy.deepcopy(model.state_dict())
        print(f"  ★ New best (val loss: {best_val:.4f})")

print(f"Done in {(time.time()-t0)/60:.1f} min. Best val loss: {best_val:.4f}")

## 8. Loss Curves

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, len(train_losses)+1), train_losses, "b-o", label="Train", markersize=4)
ax.plot(range(1, len(val_losses)+1), val_losses, "r-o", label="Val", markersize=4)
ax.set(xlabel="Epoch", ylabel="Loss", title="Fine-Tuning Loss (v2 → v3)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

## 9. Test Evaluation — Precision / Recall / F1

In [ ]:
def box_iou(b1, b2):
    x1, y1 = max(b1[0], b2[0]), max(b1[1], b2[1])
    x2, y2 = min(b1[2], b2[2]), min(b1[3], b2[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (b1[2]-b1[0])*(b1[3]-b1[1])
    a2 = (b2[2]-b2[0])*(b2[3]-b2[1])
    return inter / (a1+a2-inter) if (a1+a2-inter) > 0 else 0

@torch.no_grad()
def evaluate_test(model, loader, iou_thresh=0.5, conf=0.5):
    model.eval()
    tp_d, fp_d, fn_d = defaultdict(int), defaultdict(int), defaultdict(int)
    for images, targets in loader:
        outs = model([i.to(DEVICE) for i in images])
        for out, tgt in zip(outs, targets):
            gt_boxes, gt_labels = tgt["boxes"].numpy(), tgt["labels"].numpy()
            mask = out["scores"].cpu().numpy() >= conf
            preds = list(zip(out["boxes"].cpu().numpy()[mask], out["labels"].cpu().numpy()[mask], out["scores"].cpu().numpy()[mask]))
            matched = set()
            for pb, pl, ps in sorted(preds, key=lambda x: -x[2]):
                best_iou, best_i = 0, -1
                for gi, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                    if gi in matched or gl != pl: continue
                    iou = box_iou(pb, gb)
                    if iou > best_iou: best_iou, best_i = iou, gi
                if best_iou >= iou_thresh:
                    tp_d[int(pl)] += 1; matched.add(best_i)
                else:
                    fp_d[int(pl)] += 1
            for gi, gl in enumerate(gt_labels):
                if gi not in matched: fn_d[int(gl)] += 1

    print(f"{'Class':<30} {'Prec':>8} {'Rec':>8} {'F1':>8}")
    print("-"*56)
    all_cls = set(list(tp_d)+list(fp_d)+list(fn_d))
    for c in sorted(all_cls):
        tp, fp, fn = tp_d[c], fp_d[c], fn_d[c]
        p = tp/(tp+fp) if tp+fp else 0
        r = tp/(tp+fn) if tp+fn else 0
        f1 = 2*p*r/(p+r) if p+r else 0
        print(f"{CLASS_NAMES.get(c, c):<30} {p:>8.3f} {r:>8.3f} {f1:>8.3f}")

if best_state:
    model.load_state_dict(best_state)
print("\n=== Test Set Evaluation ===\n")
evaluate_test(model, test_loader)

## 10. Save & Download dental_model_v3.pth

Upload to Railway: `models/dental_model_v3.pth` (backend auto-prefers v3 over v2).

In [ ]:
OUTPUT = os.path.join(WORK_DIR, "dental_model_v3.pth")
checkpoint = {
    "model_state_dict": best_state if best_state else model.state_dict(),
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "category_map": CATEGORY_MAP,
    "roboflow_categories": raw_categories,
    "image_size": IMAGE_SIZE,
    "best_val_loss": best_val,
    "train_losses": train_losses,
    "val_losses": val_losses,
    "source_model": "dental_model_v2.pth",
    "dataset": "roboflow/opg-unbmz/dental-x-ray-panoramic/v2",
}
torch.save(checkpoint, OUTPUT)
print(f"Saved: {OUTPUT} ({os.path.getsize(OUTPUT)/1e6:.1f} MB)")
files.download(OUTPUT)
print("\n✅ Done! Upload dental_model_v3.pth to models/ on Railway.")